In [121]:
import pandas as pd

df = pd.read_csv('data/raw/day_approach_maskedID_timeseries.csv')
df = df.sort_values(['Athlete ID', 'Date'])
df['day'] = df.groupby('Athlete ID').cumcount()   # 0, 1, 2, ... within each athlete

feature_cols = ['total km', 'nr. sessions', 'km Z3-4', 'km Z5-T1-T2',
                'km sprinting', 'strength training', 'hours alternative',
                'perceived exertion', 'perceived trainingSuccess',
                'perceived recovery']

rolled = (df.groupby('Athlete ID')[feature_cols]
            .rolling(window=7)
            .agg(['mean', 'std', 'max'])
            .reset_index())

rolled.columns = ['_'.join(c).strip('_') for c in rolled.columns]
rolled = rolled.rename(columns={'level_1': 'orig_row'})   # row label in df
rolled = rolled.dropna()

rolled['start'] = df.loc[rolled['orig_row'], 'day'].to_numpy()
rolled['stop'] = rolled['start'] + 1
rolled['injury'] = df.loc[rolled['orig_row'], 'injury'].to_numpy()

rolled = rolled.drop(columns='orig_row')
print(rolled.head())

    Athlete ID  total km_mean  total km_std  total km_max  nr. sessions_mean  \
6            0       3.171429      6.220856          16.4           0.714286   
7            0       3.085714      6.182618          16.4           0.714286   
8            0       3.085714      6.182618          16.4           0.714286   
9            0       3.085714      6.182618          16.4           0.714286   
10           0       5.600000      8.023299          17.6           0.857143   

    nr. sessions_std  nr. sessions_max  km Z3-4_mean  km Z3-4_std  \
6           0.487950               1.0      1.428571     3.779645   
7           0.487950               1.0      1.428571     3.779645   
8           0.487950               1.0      1.428571     3.779645   
9           0.487950               1.0      1.428571     3.779645   
10          0.377964               1.0      2.457143     4.273506   

    km Z3-4_max  ...  perceived exertion_max  perceived trainingSuccess_mean  \
6          10.0  ...    

GroupKFold prevents leakage
Stratification handles imbalance
combine them.

groups = Athlete ID  keeps athletes intact
y = injury (row-level) balances rare event across folds
StratifiedGroupKFold combines both

In [139]:
from sklearn.model_selection import StratifiedGroupKFold
from lifelines import CoxTimeVaryingFitter
x = rolled.drop(columns=['Athlete ID', 'start','stop', 'injury'])
sgkf = StratifiedGroupKFold(n_splits=3)
for fold, (train_idx, test_idx) in enumerate(sgkf.split(x, rolled['injury'], rolled['Athlete ID'])):
  #  print(f"Fold {fold}:")
    #print(f"Train {rolled.iloc[train_idx]['Athlete ID']}")
    #print(f"Test {rolled.iloc[test_idx]['Athlete ID']}")
    train_athletes = set(rolled.iloc[train_idx]['Athlete ID'])
    test_athletes = set(rolled.iloc[test_idx]['Athlete ID'])
    overlap = train_athletes & test_athletes
   # print(f"  Leak?: {len(overlap) > 0}")
    #print(   rolled.iloc[train_idx].var())
   # print(   rolled.iloc[train_idx].corr())
    #print(   rolled.iloc[train_idx]['injury'].sum())
    #print(x.iloc[train_idx])
    training_data = rolled.iloc[train_idx]    
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        df=training_data,
        id_col="Athlete ID",
        event_col="injury",
        start_col="start",
        stop_col="stop",
        #show_progress=True,
    )
    print(ctv.summary)
    
 #   print(f"C-index: {ctv.concordance_index_:.3f}")
    

                                    coef  exp(coef)  se(coef)  coef lower 95%  \
covariate                                                                       
total km_mean                  -0.042469   0.958420  0.036248       -0.113513   
total km_std                    0.170188   1.185528  0.059274        0.054014   
total km_max                   -0.058458   0.943218  0.029378       -0.116038   
nr. sessions_mean               0.633657   1.884489  0.466932       -0.281514   
nr. sessions_std               -0.009369   0.990674  0.632314       -1.248682   
nr. sessions_max                0.148300   1.159861  0.249608       -0.340922   
km Z3-4_mean                   -0.530600   0.588252  0.344128       -1.205078   
km Z3-4_std                     0.502137   1.652249  0.697507       -0.864951   
km Z3-4_max                    -0.094386   0.909932  0.223416       -0.532273   
km Z5-T1-T2_mean                0.289611   1.335907  0.252898       -0.206060   
km Z5-T1-T2_std             